<a href="https://colab.research.google.com/github/izzat-ai/learning-ai/blob/main/projects/credit_scoring_project/credit_scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import sklearn

In [2]:
df = pd.read_csv("/content/drive/MyDrive/AI learning/ai_project_2026/adult/adult11.csv")
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,salary
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [3]:
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education-num',
       'marital-status', 'occupation', 'relationship', 'race', 'gender',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
       'salary'],
      dtype='object')

In [6]:
df.shape

(48842, 15)

In [7]:
df.isnull().sum()

,0
age,0
workclass,0
fnlwgt,0
education,0
education-num,0
marital-status,0
occupation,0
relationship,0
race,0
gender,0


In [4]:
df.duplicated().sum()

np.int64(52)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       48842 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      48842 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   gender          48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48842 non-null  object
 14  salary          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


- salary ustuni object turida ekan

In [9]:
df.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


- 90 yoshli mijozlar bank uchun outlier bo'lishi mumkin chunki bu yoshda odatda kredit olinmiydi
- capital-gain ustunidagi max qiymat 99999 ga teng , bu shubhali
- hours-per-week ustunidagi max qiymat ham oulier

In [10]:
# ma'lumotlar ichida ? belgisi bor edi , shular nechtaligini aniqlash
for col in df.columns:
    if df[col].dtype == 'object':
        count = (df[col].str.strip() == '?').sum()
        if count > 0:
            print(f"{col}: {count} ta '?' bor")

workclass: 2799 ta '?' bor
occupation: 2809 ta '?' bor
native-country: 857 ta '?' bor


- agar insonning ish joyi turi noma'lum bo'lsa, katta ehtimol bilan uning kasbi ham noma'lum bo'ladi . Shuning uchun ham workclass va occupation ustunidagi "?" lar deyarli bir xil
- Bank sohasida mijozning kasbi va ish joyi turi uning kredit layoqatini baholashda eng muhim faktorlardan hisoblanadi. Shuning uchun bularni o'chirib tashlamiymiz

In [12]:
# matnli ustunlardagi unikal qiymatlarni aniqlash
print("Oilaviy ahvol ustunidagi unikal qiymatlari:", df['marital-status'].unique())

Oilaviy ahvol ustunidagi unikal qiymatlari: ['Never-married' 'Married-civ-spouse' 'Widowed' 'Divorced' 'Separated'
 'Married-spouse-absent' 'Married-AF-spouse']


- bu kategoriyalarni 2 guruhga married va single qilishimiz mumkin

In [15]:
# odatda kasb egalari haftasiga 70 soat ishlaydi
# 70 soatdan ko'p ishlaydigan mijozlarni aniqlash
(df['hours-per-week']>70).sum()

np.int64(774)

- 774 ta mijoz haftasiga 70 soatdan ko'p ishlaydi . Bularning daromadi juda yuqori yoki moliyaviy barqarorligi past bo'lishi mumkin (bir nechta joyda ishledigan bo'lishi mumkin)

In [18]:
# bashorat qilmoqchi bo'lgan ustunimiz taqsimotini aniqlash
print(df['salary'].value_counts())

salary
<=50K    37155
>50K     11687
Name: count, dtype: int64


In [19]:
# foiz ko'rinishida
df['salary'].value_counts(normalize=True) * 100

,proportion
salary,
<=50K,76.071823
>50K,23.928177


- ushbu bashorat qilishimiz kerak bo'lgan ustunimizdagi ma'lumotlar teng emas , ya'ni balansda emas
- bu kelajakda modelimizni noto'g'ri bashorat qilishiga olib keladi . Daromadi kam degan javobga og'ib ketadi

##Data spliting : train/test set

In [20]:
from sklearn.model_selection import train_test_split

# train va test setlarga bo'lish (80%-train, 20%-test)
# y-ustunidagi 76/24 nisbati saqlanib qolgan holda
train_set, test_set = train_test_split(df, test_size=0.2, random_state=42, stratify=df['salary'])

print("train_set hajmi:", train_set.shape)
print("test_set hajmi:", test_set.shape)

train_set hajmi: (39073, 15)
test_set hajmi: (9769, 15)


## Data Cleaning . train_set

In [22]:
# yuqorida duplikatlarni aniqlagan edik
# duplikatlarni o'chirish
train_set = train_set.drop_duplicates()
train_set.shape

(39040, 15)

In [27]:
# yuqori "?" belgisi bor ekanligini ko'rgandik , shular nechtaligini aniqlash
miss_val_train = (train_set == '?').sum()
miss_val_train[miss_val_train > 0]

,0
workclass,2220
occupation,2230
native-country,681
